In [ ]:
# ============================================================
# HYBRID BAGGED ENSEMBLE WITH CALIBRATION
# Google Colab – полный код с обучением
# ============================================================

!pip install pytorch-tabnet -q

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, fbeta_score, matthews_corrcoef
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# ============================================================
# 1. ЗАГРУЗКА И ПРЕДОБРАБОТКА ДАННЫХ
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# Пути к файлам (измените на свои)
DATA_DIR = ''
trans = pd.read_csv(f'{DATA_DIR}/train_transaction.csv')
identity = pd.read_csv(f'{DATA_DIR}/train_identity.csv')
df = pd.merge(trans, identity, on='TransactionID', how='left')

print(f"Dataset: {df.shape}")
print(f"Fraud rate: {df['isFraud'].mean():.4f}")

# Time-based split (60/20/20)
df_sorted = df.sort_values('TransactionDT').reset_index(drop=True)
n = len(df_sorted)
train_end = int(n * 0.60)
valid_end = int(n * 0.80)

train_df = df_sorted.iloc[:train_end].copy()
valid_df = df_sorted.iloc[train_end:valid_end].copy()
test_df = df_sorted.iloc[valid_end:].copy()

print(f"Train: {train_df.shape}, Valid: {valid_df.shape}, Test: {test_df.shape}")

# Упрощённая предобработка (базовые фичи)
def basic_preprocessing(train, valid, test, target='isFraud'):
    # Определяем колонки для удаления по train
    missing_ratio = train.isnull().mean()
    cols_to_drop = missing_ratio[missing_ratio > 0.8].index.tolist()
    for col in ['TransactionID']:
        if col in cols_to_drop:
            cols_to_drop.remove(col)
    
    # Удаляем
    train = train.drop(columns=cols_to_drop, errors='ignore')
    valid = valid.drop(columns=cols_to_drop, errors='ignore')
    test = test.drop(columns=cols_to_drop, errors='ignore')
    
    # Разделяем X/y
    X_train = train.drop(columns=[target])
    y_train = train[target]
    X_valid = valid.drop(columns=[target])
    y_valid = valid[target]
    X_test = test.drop(columns=[target])
    y_test = test[target]
    
    # Категориальные колонки
    cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
    num_cols = [c for c in X_train.columns if c not in cat_cols]
    
    # Заполняем пропуски
    medians = X_train[num_cols].median(numeric_only=True)
    X_train[num_cols] = X_train[num_cols].fillna(medians)
    X_valid[num_cols] = X_valid[num_cols].fillna(medians)
    X_test[num_cols] = X_test[num_cols].fillna(medians)
    
    X_train[cat_cols] = X_train[cat_cols].fillna('missing')
    X_valid[cat_cols] = X_valid[cat_cols].fillna('missing')
    X_test[cat_cols] = X_test[cat_cols].fillna('missing')
    
    # Ordinal encoding
    if cat_cols:
        encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
        X_valid[cat_cols] = encoder.transform(X_valid[cat_cols])
        X_test[cat_cols] = encoder.transform(X_test[cat_cols])
    
    return X_train, y_train, X_valid, y_valid, X_test, y_test

X_train, y_train, X_valid, y_valid, X_test, y_test = basic_preprocessing(
    train_df, valid_df, test_df
)

print(f"X_train: {X_train.shape}, X_valid: {X_valid.shape}, X_test: {X_test.shape}")

# ============================================================
# 2. СОЗДАНИЕ RECENCY ВЕСОВ
# ============================================================

def make_recency_weights(n_old, n_new, recent_weight=3.0):
    return np.concatenate([np.ones(n_old), np.ones(n_new) * recent_weight])

# Веса для всей тренировочной выборки (train + valid)
X_full = np.vstack([X_train.values, X_valid.values])
y_full = np.concatenate([y_train.values, y_valid.values])

n_old = len(X_train)
n_new = len(X_valid)
recency_weights_2 = make_recency_weights(n_old, n_new, recent_weight=2.0)
recency_weights_3 = make_recency_weights(n_old, n_new, recent_weight=3.0)

print(f"Recency weights created: {n_old} old, {n_new} new")

# ============================================================
# 3. ОБУЧЕНИЕ МОДЕЛЕЙ ДЛЯ BAGGING
# ============================================================

scale_pos_weight = len(y_full[y_full==0]) / max(1, len(y_full[y_full==1]))
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

# LightGBM модели (3 штуки с разными seed)
lgb_models = []
lgb_seeds = [404, 101, 303]
lgb_weights = [recency_weights_3, recency_weights_2, recency_weights_3]

for seed, weight in zip(lgb_seeds, lgb_weights):
    print(f"Training LightGBM with seed {seed}...")
    model = LGBMClassifier(
        n_estimators=1200,
        learning_rate=0.024,
        num_leaves=96,
        min_child_samples=75,
        subsample=0.84,
        colsample_bytree=0.84,
        reg_alpha=0.05,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1
    )
    model.fit(X_full, y_full, sample_weight=weight)
    lgb_models.append(model)

# XGBoost модели (3 штуки с разными seed)
xgb_models = []
xgb_seeds = [101, 202, 303]
xgb_weights = [recency_weights_2, recency_weights_3, recency_weights_2]

for seed, weight in zip(xgb_seeds, xgb_weights):
    print(f"Training XGBoost with seed {seed}...")
    model = XGBClassifier(
        n_estimators=983,
        learning_rate=0.056,
        max_depth=7,
        subsample=0.715,
        colsample_bytree=0.95,
        scale_pos_weight=scale_pos_weight,
        reg_alpha=1.93,
        reg_lambda=5.28,
        random_state=seed,
        n_jobs=-1,
        eval_metric='logloss',
        verbosity=0
    )
    model.fit(X_full, y_full, sample_weight=weight)
    xgb_models.append(model)

# ============================================================
# 4. ПОЛУЧЕНИЕ ПРЕДСКАЗАНИЙ
# ============================================================

X_test_np = X_test.values.astype(np.float32)

# LightGBM предсказания
lgb_preds = []
for model in lgb_models:
    proba = model.predict_proba(X_test_np)[:, 1]
    lgb_preds.append(proba)

# XGBoost предсказания
xgb_preds = []
for model in xgb_models:
    proba = model.predict_proba(X_test_np)[:, 1]
    xgb_preds.append(proba)

# ============================================================
# 5. BAGGING + WEIGHTED BLEND
# ============================================================

lgb_avg = np.mean(lgb_preds, axis=0)
xgb_avg = np.mean(xgb_preds, axis=0)

# Weighted blend (веса из старой оптимизации)
hybrid_proba = 0.53 * xgb_avg + 0.47 * lgb_avg

# ============================================================
# 6. ISOTONIC CALIBRATION (на валидации)
# ============================================================

# Получаем предсказания на валидации для калибровки
X_valid_np = X_valid.values.astype(np.float32)

# LightGBM на валидации
lgb_valid_preds = []
for model in lgb_models:
    proba = model.predict_proba(X_valid_np)[:, 1]
    lgb_valid_preds.append(proba)

# XGBoost на валидации
xgb_valid_preds = []
for model in xgb_models:
    proba = model.predict_proba(X_valid_np)[:, 1]
    xgb_valid_preds.append(proba)

lgb_avg_valid = np.mean(lgb_valid_preds, axis=0)
xgb_avg_valid = np.mean(xgb_valid_preds, axis=0)
hybrid_proba_valid = 0.53 * xgb_avg_valid + 0.47 * lgb_avg_valid

# Обучаем калибровку
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(hybrid_proba_valid, y_valid.values)

# Применяем к тесту
hybrid_proba_calibrated = iso.transform(hybrid_proba)

# ============================================================
# 7. ФИНАЛЬНЫЕ МЕТРИКИ
# ============================================================

def evaluate(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        'PR-AUC': average_precision_score(y_true, y_proba),
        'ROC-AUC': roc_auc_score(y_true, y_proba),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F2': fbeta_score(y_true, y_pred, beta=2),
        'MCC': matthews_corrcoef(y_true, y_pred)
    }

print("\n" + "="*50)
print("HYBRID BAGGED ENSEMBLE WITH CALIBRATION")
print("="*50)

metrics = evaluate(y_test.values, hybrid_proba_calibrated)
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# ============================================================
# 8. СРАВНЕНИЕ С ОТДЕЛЬНЫМИ МОДЕЛЯМИ
# ============================================================

print("\n" + "="*50)
print("COMPARISON WITH INDIVIDUAL MODELS")
print("="*50)

# Лучший LightGBM (seed 404)
best_lgb = lgb_models[0]
best_lgb_proba = best_lgb.predict_proba(X_test_np)[:, 1]
best_lgb_metrics = evaluate(y_test.values, best_lgb_proba)

# Лучший XGBoost (seed 101)
best_xgb = xgb_models[0]
best_xgb_proba = best_xgb.predict_proba(X_test_np)[:, 1]
best_xgb_metrics = evaluate(y_test.values, best_xgb_proba)

print("Best LightGBM (seed 404):")
for k, v in best_lgb_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nBest XGBoost (seed 101):")
for k, v in best_xgb_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nHybrid Bagged Ensemble (calibrated):")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

print("\n✅ Done! Hybrid Bagged Ensemble with Calibration is ready.")